# 01 — Explore the RTS data

**Today’s goal:** load Route 7, understand what a row means, and identify questions to resolve before calculating travel times.

Run cells from top to bottom with **Shift+Enter**. Choose your project’s **.venv** kernel in VS Code. This notebook reads the original files; it does not change them.

By the end, you should have a data overview, a few plots, and three cleaning questions. **We are not fitting a model yet.**

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Working directory:", Path.cwd())

## 1. Find the project and load Route 7

The next cell works when you launch from the repository root or from its `notebooks` folder. If it cannot find the data, open the extracted repository folder in VS Code first.

`read_excel` loads the workbook into a **dataframe**: a table with rows and columns.

In [ ]:
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (p for p in candidates
     if (p / "data" / "raw" / "RTS_Route7_March2023.xlsx").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open the repository folder in VS Code. Expected "
        "data/raw/RTS_Route7_March2023.xlsx under that folder."
    )

DATA_DIR = PROJECT_ROOT / "data" / "raw"
data_path = DATA_DIR / "RTS_Route7_March2023.xlsx"
raw = pd.read_excel(data_path, engine="openpyxl")
df = raw.copy()  # Work on a copy; keep raw and the workbook unchanged.
print("Project root:", PROJECT_ROOT)
print("Loaded:", data_path.name)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head())

**Discuss:** one row is a recorded bus stop visit, not a whole journey. Which columns tell you the bus, stop, date, and arrival time? Could the same bus visit the same stop more than once in a day?

In [ ]:
print("Columns:")
print(df.columns.tolist())
print("\nColumn types and non-missing counts:")
df.info()

quality = pd.DataFrame({
    "type": df.dtypes.astype(str),
    "missing_rows": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True),
})
display(quality)

## 2. Check dates and arrival times

Dates and times come from Excel and need checking. We preserve the original fields and create separate inspection columns. `errors="coerce"` marks unparseable values as missing instead of silently guessing.

This first inspection assumes `Date` is the calendar date. That assumption still needs checking for overnight service.

In [ ]:
df["service_date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
clock_text = df["Arrive"].astype("string").str.strip()
clock_delta = pd.to_timedelta(clock_text, errors="coerce")
valid_clock = clock_delta.ge(pd.Timedelta(0)) & clock_delta.lt(pd.Timedelta(days=1))
df["arrival_timestamp"] = df["service_date"] + clock_delta.where(valid_clock)
df["recorded_weekday_matches"] = (
    df["Day of Week"].astype("string").str.strip().str.lower()
    == df["service_date"].dt.day_name().str.lower()
)

print("Unparsed dates:", df["service_date"].isna().sum())
print("Missing/invalid arrival timestamps:", df["arrival_timestamp"].isna().sum())
print("Weekday mismatches or missing labels:",
      (~df["recorded_weekday_matches"].fillna(False)).sum())
print("Observed range:", df["service_date"].min(), "to", df["service_date"].max())
print("Distinct dates:", df["service_date"].nunique())
print("Distinct buses:", df["Bus"].nunique())
print("Distinct stop IDs:", df["Stop ID"].nunique())
display(df[["Date", "Arrive", "arrival_timestamp"]].head())

### What dates and hours do we actually observe?

These charts count **recorded rows**, not completed trips or buses per hour. A high count can reflect coverage, repeated records, or service activity. Missing dates do not prove that no buses operated.

In [ ]:
date_counts = df.groupby("service_date").size().rename("recorded_rows")
display(date_counts.to_frame())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
date_counts.rename(index=lambda x: x.strftime("%m-%d")).plot.bar(ax=axes[0])
axes[0].set(title="Recorded stop events by observed date", xlabel="Date", ylabel="Rows")
df["arrival_timestamp"].dt.hour.value_counts().sort_index().plot.bar(ax=axes[1])
axes[1].set(title="Recorded stop events by hour", xlabel="Hour (recorded local time)", ylabel="Rows")
plt.tight_layout()
plt.show()

## 3. Investigate repeated records

A repeated date/bus/stop/arrival combination is a **flag**, not an instruction to delete the row. Compare the other columns and discuss possible explanations first.

In [ ]:
event_key = ["Date", "Bus", "Stop ID", "Arrive"]
print("Exact duplicate rows (all original columns):", raw.duplicated().sum())
print("Excess repeated event keys:", raw.duplicated(event_key).sum())
repeated = raw.loc[raw.duplicated(event_key, keep=False)]
display(repeated.sort_values(["Date", "Bus", "Arrive", "Stop ID"]).head(12))

## 4. Follow one bus on one day

The table below is a viewing aid. A bus-day can contain several journeys. Look for stop-order restarts, repeated times, terminal visits, and large gaps. We have **not** assigned trip IDs or accepted any travel-time labels.

If two events have the same timestamp, sorting alone cannot establish their true order.

In [ ]:
example_date = df["service_date"].dropna().min()
example_bus = df.loc[df["service_date"].eq(example_date), "Bus"].dropna().iloc[0]
bus_day = df.loc[
    df["service_date"].eq(example_date) & df["Bus"].eq(example_bus)
].sort_values("arrival_timestamp", kind="stable").copy()

print("Example date:", example_date.date(), "| Bus:", example_bus)
display(bus_day[["Bus", "Stop ID", "Stop Name", "Stop Seq ID", "Arrive",
                 "Passenger On", "Passenger Off", "Passenger Load"]].head(35))

## 5. Optional: inventory all four route files

This only summarizes the files. It does not combine routes into one cleaned training set. The larger row counts are raw stop records, not verified journeys.

In [ ]:
route_files = {
    "7": "RTS_Route7_March2023.xlsx",
    "3": "RTS_Route3_Spring2023.xlsx",
    "11": "RTS_Route11_Spring2023.xlsx",
    "711": "RTS_Route711_Spring2023.xlsx",
}
inventory = []
for route, filename in route_files.items():
    path = DATA_DIR / filename
    if not path.exists():
        print("Not present; skipping:", filename)
        continue
    table = raw if route == "7" else pd.read_excel(path, engine="openpyxl")
    dates = pd.to_datetime(table["Date"], errors="coerce").dt.normalize()
    inventory.append({
        "route": route, "raw_rows": len(table), "observed_dates": dates.nunique(),
        "first_date": dates.min(), "last_date": dates.max(),
        "buses": table["Bus"].nunique(), "stops": table["Stop ID"].nunique(),
    })
display(pd.DataFrame(inventory))

## 6. Your group’s notes — edit this cell

Replace the prompts with your answers. Use specific counts or examples.

1. **What does one row represent?** Your answer:
2. **What dates and buses do we observe?** Your answer:
3. **What did you notice about missing values or repeated records?** Your answer:
4. **Where might one journey end and another begin?** Give one example:
5. **What three questions must we resolve before making stop pairs?** Your answer:

### Before your check-in

- Run the notebook from top to bottom and save it.
- Bring your data overview, the two charts, and three cleaning questions.
- Do not remove repeated rows or fit models without explaining your decisions.

**Next milestone:** a reviewed cleaning plan and five manually verified consecutive-stop pairs. Later notebooks will cover pairing, chronological validation, and models; they have not been implemented in this starter.

**Data credit:** Richard Borzi, University of Florida; supplied by Gainesville RTS. [Route 7 source](https://zenodo.org/records/10034933), [Route 3](https://zenodo.org/records/10034688), [Route 11](https://zenodo.org/records/10034723), [Route 711](https://zenodo.org/records/10034622). Datasets are CC BY 4.0. Keep these credits when sharing derived data.